We now prepare the dataset for the experiments

- First, for each ontology and question_type we randomly sample QUESTION_N_SAMPLES using seed equal to SEED
- For every row, the row ontology is considered the "test" ontology:
    - we list the "training ontologies" (ie different to the one in the row) with questions of the same type as the question_type in the row.

- We select up to MAX_FEW_SHOTS to be used as examples in in-context learning.
    - If the number K of ontologies with the target question type is less than MAX_FEW_SHOTS,
        - first, we select one question for each of the K ontologies
        - second, se sample the remaining (MAX_FEW_SHOTS-K) examples from the remaining rows (excluding the questions already sample), with replacement
        

In [45]:
import pandas as pd
import numpy as np

In [46]:
# SEED = 123
# SEED = 234
# SEED = 345
# SEED = 444
# SEED = 555
# SEED = 666
# SEED = 777
# SEED = 999
# SEED = 888
SEED = 357

QUESTION_N_SAMPLES = 10

MAX_FEW_SHOTS = 10



In [47]:
# read data produce by notebook at the previous step (cleaning)


df = pd.read_csv("../dataset/qgenllm-updated-2-filtered-min_q.csv", sep=";")
print(df.shape)

print(df.groupby(["ontology", "question_type"]).size())


(489130, 11)
ontology      question_type                   
AWO           Definition                             69
              What-1-part-1-rel                     100
              What-2-part-1-rel                     192
              What-2-part-1-rel-quant-only           58
              What-2-part-1-rel-quant-some          134
              Yes-No-2-part-1-rel                   300
              Yes-No-2-part-1-rel+1-quant-only       72
              Yes-No-2-part-1-rel+1-quant-some      228
BioTop        Definition                            149
              What-1-part-1-rel                    6854
              What-2-part-1-rel                    6854
              What-2-part-1-rel-quant-only         6160
              What-2-part-1-rel-quant-some          694
              Yes-No-2-part-1-rel                 20562
              Yes-No-2-part-1-rel+1-quant-only    18480
              Yes-No-2-part-1-rel+1-quant-some     2082
CopyrightAll  What-1-part-1-rel             

In [48]:
# sampling here
df_questions = df.groupby(["ontology", "question_type"]).sample(n=QUESTION_N_SAMPLES, random_state=SEED)
print(f"Shape after sampling {QUESTION_N_SAMPLES} questions per question type and ontology:")
print(df_questions.shape)

print("SAMPLE")
display(df_questions.sample(20))

print()
print("ONTOLOGIES")
print(df_questions.ontology.unique())
print()

print()
print("QUESTION TYPES")
for i, t in enumerate(df_questions.question_type.unique()):
    print(f"{i+1}. {t}")


for ontology, gdf in df_questions.groupby("ontology"):
    print(f"Ontology: {ontology}")
    print(f"Question types: {gdf['question_type'].unique()}")
    print(f"N question types: {gdf['question_type'].nunique()}")
    print(f"Total questions: {gdf.shape[0]}")
    print(10 * "-")

# df_questions.to_csv(f"../dataset/qgenllm-ds-seed-{SEED}.csv", sep=";", index=False)

Shape after sampling 10 questions per question type and ontology:
(500, 11)
SAMPLE


,axiom_pattern,axiom,template,question,question_type,op_category,ontology,file,question_type_short,op_category_short,row_id
79727,[X] SubclassOf([prop] only [Y]),TemporaryClosureType SubclassOf(hasPriceSpecif...,Does [X] [prop/*OP_HAS_NOUNS-infinitive] that ...,Does a type of temporary closure have currency...,Yes-No-2-part-1-rel+1-quant-only,OP_HAS_NOUNS,Cultural-On,Yes-No-2-part-1-rel+1-quant-only.csv,yn2p1r1_1_qo,has_nouns,79727
721,[X] SubclassOf([prop] some [Y]),Apple SubclassOf(part-of some Plant),[X] is [prop/*OP_IS_NOUNS_PREP] [Y]. True or f...,An apple is part of a plant. True or false?,Yes-No-2-part-1-rel,OP_IS_NOUNS_PREP,AWO,Yes-No-2-part-1-rel.csv,yn2p1r,is_noun_prep,721
400,"[Z] SubclassOf([X]),[Z] SubclassOf([prop] only...","Camelopard SubclassOf(Live-on only Habitat),Ca...",Which [X/noArticle] [prop/*OP_VERB_PREP-third]...,Which terrestrial animal lives on only a habitat?,What-2-part-1-rel-quant-only,OP_VERB_PREP,AWO,What-2-part-1-rel-quant-only.csv,w2p1r_qo,verb_prep,400
1088,[X] SubclassOf([prop] some [Y]),Twig SubclassOf(proper-part-of some Plant),[X] is [prop/*OP_IS_NOUNS_PREP] some [Y]. True...,A twig is proper part of some plant. True or f...,Yes-No-2-part-1-rel+1-quant-some,OP_IS_NOUNS_PREP,AWO,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,is_noun_prep,1088
68745,[Z] SubclassOf([prop] only [Y]),CulturalLandscapeAsset SubclassOf(hasSite only...,Which [X/noArticle] [prop/*OP_HAS_NOUNS-third]...,Which cultural entity has a site that is only ...,What-2-part-1-rel-quant-only,OP_HAS_NOUNS,Cultural-On,What-2-part-1-rel-quant-only.csv,w2p1r_qo,has_nouns,68745
489076,"[Z] SubclassOf([X]),[Z] SubclassOf([prop] some...","hasConjunct SubclassOf(Relation),hasConjunct S...",Which [X/noArticle] [prop/*OP_HAS_NOUNS-third]...,Which dependency relation has a parent that is...,What-2-part-1-rel,OP_HAS_NOUNS,olia,What-2-part-1-rel.csv,w2p1r,has_nouns,489076
70,[X] SubclassOf([prop] some [Y]),Warthog SubclassOf(eats some Grass),What does [X] [prop/*OP_VERB-infinitive]?,What does a warthog eat?,What-1-part-1-rel,OP_VERB,AWO,What-1-part-1-rel.csv,w1p1r,verb,70
68534,[Z] SubclassOf([prop] only [Y]),Collection SubclassOf(hasMember only CulturalE...,Which [X/noArticle] [prop/*OP_HAS_NOUNS-third]...,Which cultural entity has a member that is onl...,What-2-part-1-rel-quant-only,OP_HAS_NOUNS,Cultural-On,What-2-part-1-rel-quant-only.csv,w2p1r_qo,has_nouns,68534
63199,[X] SubclassOf([prop] only [Y]),PerformInPublic SubclassOf(location only Publi...,True or false: [X] [prop/*OP_HAS_NOUNS-third] ...,True or false: A perform in public has a locat...,Yes-No-2-part-1-rel,OP_HAS_NOUNS,CopyrightAll,Yes-No-2-part-1-rel.csv,yn2p1r,has_nouns,63199
147285,[X] SubclassOf([prop] some [Y]),AsparagusTopping SubclassOf(hasSpiciness some ...,Does [X] [prop/*OP_HAS_NOUNS-infinitive] that ...,Does an asparagus topping have a spiciness tha...,Yes-No-2-part-1-rel+1-quant-some,OP_HAS_NOUNS,Pizza,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,has_nouns,147285



ONTOLOGIES
['AWO' 'BioTop' 'CopyrightAll' 'Cultural-On' 'Pizza' 'Stuff' 'olia']


QUESTION TYPES
1. Definition
2. What-1-part-1-rel
3. What-2-part-1-rel
4. What-2-part-1-rel-quant-only
5. What-2-part-1-rel-quant-some
6. Yes-No-2-part-1-rel
7. Yes-No-2-part-1-rel+1-quant-only
8. Yes-No-2-part-1-rel+1-quant-some
Ontology: AWO
Question types: ['Definition' 'What-1-part-1-rel' 'What-2-part-1-rel'
 'What-2-part-1-rel-quant-only' 'What-2-part-1-rel-quant-some'
 'Yes-No-2-part-1-rel' 'Yes-No-2-part-1-rel+1-quant-only'
 'Yes-No-2-part-1-rel+1-quant-some']
N question types: 8
Total questions: 80
----------
Ontology: BioTop
Question types: ['Definition' 'What-1-part-1-rel' 'What-2-part-1-rel'
 'What-2-part-1-rel-quant-only' 'What-2-part-1-rel-quant-some'
 'Yes-No-2-part-1-rel' 'Yes-No-2-part-1-rel+1-quant-only'
 'Yes-No-2-part-1-rel+1-quant-some']
N question types: 8
Total questions: 80
----------
Ontology: CopyrightAll
Question types: ['What-1-part-1-rel' 'What-2-part-1-rel' 'What-2-part-1-rel

SAMPLE QUESTIONS

In [49]:
# alias here
df = df_questions

# build dictionary: question_type -> list of ontologies that have that question type
qt2onto = {}
for qt, gdf in df.groupby("question_type"):
    ontologies = gdf['ontology'].unique().tolist()
    qt2onto[qt] = ontologies

for k, v in qt2onto.items():
    print(f"Question type: {k}")
    print(f"  Ontologies: {v}")
    assert len(v) > 0, f"Question type {k} has no ontologies"
    print("-")

print()

rng = np.random.default_rng(SEED)
df["seed"] = rng.integers(0, 2**32-1, size=df.shape[0])



# prepare a list of up to MAX_FEW_SHOTS

def select_ontos(row):
    qt = row["question_type"]
    # select other ontos by removing the ontology in the current row
    other_ontos = [o for o in qt2onto[qt] if o != row["ontology"]]
    assert len(other_ontos) > 0, f"Question type {qt} has only one ontology, cannot select few-shots"
    rng = np.random.default_rng(row["seed"])
    # shuffle other_ontos
    other_ontos = rng.permutation(other_ontos)
    # now sample with replacement up to MAX_FEW_SHOTS
    remaining_ontos = rng.choice(other_ontos, size=MAX_FEW_SHOTS-len(other_ontos), replace=True)
    selected_ontos = list(other_ontos) + list(remaining_ontos)
    return selected_ontos

df["q_from_os"] = df.apply(select_ontos, axis=1)

display(df)


Question type: Definition
  Ontologies: ['AWO', 'BioTop', 'Cultural-On', 'Pizza', 'Stuff']
-
Question type: What-1-part-1-rel
  Ontologies: ['AWO', 'BioTop', 'CopyrightAll', 'Cultural-On', 'Pizza', 'Stuff']
-
Question type: What-2-part-1-rel
  Ontologies: ['AWO', 'BioTop', 'CopyrightAll', 'Cultural-On', 'Pizza', 'Stuff', 'olia']
-
Question type: What-2-part-1-rel-quant-only
  Ontologies: ['AWO', 'BioTop', 'CopyrightAll', 'Cultural-On', 'Pizza', 'Stuff']
-
Question type: What-2-part-1-rel-quant-some
  Ontologies: ['AWO', 'BioTop', 'CopyrightAll', 'Cultural-On', 'Pizza', 'Stuff', 'olia']
-
Question type: Yes-No-2-part-1-rel
  Ontologies: ['AWO', 'BioTop', 'CopyrightAll', 'Cultural-On', 'Pizza', 'Stuff', 'olia']
-
Question type: Yes-No-2-part-1-rel+1-quant-only
  Ontologies: ['AWO', 'BioTop', 'CopyrightAll', 'Cultural-On', 'Pizza', 'Stuff']
-
Question type: Yes-No-2-part-1-rel+1-quant-some
  Ontologies: ['AWO', 'BioTop', 'Cultural-On', 'Pizza', 'Stuff', 'olia']
-



,axiom_pattern,axiom,template,question,question_type,op_category,ontology,file,question_type_short,op_category_short,row_id,seed,q_from_os
63,[X] EquivalentTo([A]),Lion EquivalentTo(King-of-beasts),What is [X]?,What is a lion?,Definition,OP_DEFINITION,AWO,Definition.csv,def,definition,63,2685017989,"[Stuff, Pizza, BioTop, Cultural-On, Stuff, Stu..."
15,[X] SubclassOf([Y]),Apple SubclassOf(PlantParts),What is [X]?,What is an apple?,Definition,OP_DEFINITION,AWO,Definition.csv,def,definition,15,1206569505,"[Stuff, BioTop, Cultural-On, Pizza, Cultural-O..."
61,[X] EquivalentTo([A]),Camelopard EquivalentTo(Giraffe),What is [X]?,What is a camelopard?,Definition,OP_DEFINITION,AWO,Definition.csv,def,definition,61,845103760,"[Cultural-On, Stuff, BioTop, Pizza, Stuff, Bio..."
51,[X] SubclassOf([Y]),Atmosphere SubclassOf(Habitat),What is [X]?,What is an atmosphere?,Definition,OP_DEFINITION,AWO,Definition.csv,def,definition,51,1181619693,"[Stuff, Cultural-On, Pizza, BioTop, Pizza, Stu..."
50,[X] SubclassOf([Y]),Sky SubclassOf(Habitat),What is [X]?,What is a sky?,Definition,OP_DEFINITION,AWO,Definition.csv,def,definition,50,3150070397,"[Stuff, Pizza, BioTop, Cultural-On, Cultural-O..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
489119,[X] SubclassOf([prop] some [Y]),ImpersonalVerb SubclassOf(hasSemanticValency s...,[X] [prop/*OP_HAS_NOUNS-third] that is some [Y...,An impersonal verb has a semantic valency that...,Yes-No-2-part-1-rel+1-quant-some,OP_HAS_NOUNS,olia,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,has_nouns,489119,2721730794,"[Pizza, Stuff, BioTop, Cultural-On, AWO, Stuff..."
489127,[X] SubclassOf([prop] some [Y]),hasWordConjunct SubclassOf(hasChild some Token),Does [X] [prop/*OP_HAS_NOUNS-infinitive] that ...,Does a has word conjunct have a child that is ...,Yes-No-2-part-1-rel+1-quant-some,OP_HAS_NOUNS,olia,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,has_nouns,489127,1188034154,"[AWO, Cultural-On, Pizza, BioTop, Stuff, AWO, ..."
489125,[X] SubclassOf([prop] some [Y]),hasConjunct SubclassOf(hasParent some Coordina...,[X] [prop/*OP_HAS_NOUNS-third] that is some [Y...,A has conjunct has a parent that is some non i...,Yes-No-2-part-1-rel+1-quant-some,OP_HAS_NOUNS,olia,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,has_nouns,489125,2368510522,"[Stuff, AWO, Pizza, Cultural-On, BioTop, Cultu..."
489122,[X] SubclassOf([prop] some [Y]),NonspecificPronoun SubclassOf(hasSpecificity s...,[X] [prop/*OP_HAS_NOUNS-third] that is some [Y...,A nonspecific pronoun has a specificity that i...,Yes-No-2-part-1-rel+1-quant-some,OP_HAS_NOUNS,olia,Yes-No-2-part-1-rel+1-quant-some.csv,yn2p1r_1_qs,has_nouns,489122,3844441541,"[AWO, BioTop, Pizza, Cultural-On, Stuff, Stuff..."


NOW SELECT THE ACTUAL EXAMPLES

In [50]:

def select_questions_fast(dataset: pd.DataFrame) -> pd.DataFrame:
    # (ontology, question_type) -> arrays
    buckets = {}
    # Keep only the columns we need to reduce overhead
    cols = ["ontology", "question_type", "row_id", "axiom", "question"]
    base = dataset[cols]

    # Group once; converting to numpy arrays makes sampling very fast
    for (ont, qt), g in base.groupby(["ontology", "question_type"], sort=False):
        buckets[(ont, qt)] = {
            "row_id": g["row_id"].to_numpy(),
            "axiom": g["axiom"].to_numpy(),
            "question": g["question"].to_numpy(),
        }

    sample_axioms = []
    sample_questions = []

    # Iterate rows once (much faster than apply for Python-heavy logic)
    for r in dataset.itertuples(index=False):
        qt = r.question_type
        seed = r.seed
        rng = np.random.default_rng(seed)

        used = set()
        axioms = []
        qs = []

        # other ontologies containing the same question type, from which we will sample questions
        for o in r.q_from_os:
            key = (o, qt)
            b = buckets.get(key)
            if b is None or b["row_id"].size == 0:
                raise AssertionError(f"No available questions for ontology {o} and question type {qt}")

            row_ids = b["row_id"]

            # If row_id is globally unique per ontology, used-check is usually unnecessary,
            # but we keep it to match your original logic.
            if used:
                # Try a few random draws; fall back to scanning if collisions happen
                # (collisions are typically rare unless buckets are tiny)
                idx = None
                for _ in range(10):
                    j = int(rng.integers(0, row_ids.size))
                    if row_ids[j] not in used:
                        idx = j
                        break
                if idx is None:
                    # Deterministic fallback: find first not used
                    mask = ~np.isin(row_ids, list(used))
                    if not mask.any():
                        raise AssertionError(f"No available (unused) questions for ontology {o} and question type {qt}")
                    choices = np.flatnonzero(mask)
                    idx = int(choices[int(rng.integers(0, choices.size))])
            else:
                idx = int(rng.integers(0, row_ids.size))

            used.add(row_ids[idx])
            axioms.append(b["axiom"][idx])
            qs.append(b["question"][idx])

        sample_axioms.append(axioms)
        sample_questions.append(qs)

    out = dataset.copy()
    out["sample_axioms"] = sample_axioms
    out["sample_questions"] = sample_questions
    return out

df = select_questions_fast(df)
df["dataset_seed"] = SEED
print("The dataset has now shape:", df.shape)



The dataset has now shape: (500, 16)


In [51]:
display(df.groupby(["ontology", "question_type"]).size())

ontology      question_type                   
AWO           Definition                          10
              What-1-part-1-rel                   10
              What-2-part-1-rel                   10
              What-2-part-1-rel-quant-only        10
              What-2-part-1-rel-quant-some        10
              Yes-No-2-part-1-rel                 10
              Yes-No-2-part-1-rel+1-quant-only    10
              Yes-No-2-part-1-rel+1-quant-some    10
BioTop        Definition                          10
              What-1-part-1-rel                   10
              What-2-part-1-rel                   10
              What-2-part-1-rel-quant-only        10
              What-2-part-1-rel-quant-some        10
              Yes-No-2-part-1-rel                 10
              Yes-No-2-part-1-rel+1-quant-only    10
              Yes-No-2-part-1-rel+1-quant-some    10
CopyrightAll  What-1-part-1-rel                   10
              What-2-part-1-rel                   10

In [52]:

# def select_questions(dataset):
#     def sample_questions(row):
#         qt = row["question_type"]
#         used_qs = []
#         qs = []
#         axioms = []

#         for o in row["q_from_os"]:  # ontologies with the same question type
#             # for each row we do not use twice the same question 
#             iii = (dataset.ontology == o) & (dataset.question_type == qt) & (~dataset.row_id.isin(used_qs))
#             available_qs = dataset[iii]
#             if available_qs.shape[0] == 0:
#                 assert False, f"No available questions for ontology {o} and question type {qt}"
#             # sample one question randomly
#             sampled_q = available_qs.sample(n=1, random_state=row["seed"])
#             random_row = sampled_q.iloc[0]

#             qs.append(random_row["question"])
#             used_qs.append(random_row["row_id"])
#             axioms.append(random_row["axiom"])
#         return axioms, qs
    
#     dataset["sample_axioms"], dataset["sample_questions"] = zip(*dataset.apply(sample_questions, axis=1))
#     return dataset

# df = select_questions(df)

# # XXX running this cell takes too long, approx. : REWRITE

In [53]:
# print a few samples
cols = ["ontology", "question_type", "question", "axiom", "sample_questions", "sample_axioms"]


df2 = df[cols].sample(10, random_state=SEED)
# print matching sample_axioms and sample_questions
for i, r in df2.iterrows():
    print(f"Ontology: {r['ontology']}")
    print(f"Question type: {r['question_type']}")
    print(f"Original question: {r['question']}")
    print(f"Original axiom: {r['axiom']}")
    print("Sampled questions and axioms:")
    for q, a in zip(r["sample_questions"], r["sample_axioms"]):
        print(f"  - Question: {q}")
        print(f"    Axiom: {a}")
    print(20 * "-")
    print()


Ontology: Pizza
Question type: Yes-No-2-part-1-rel+1-quant-some
Original question: True or false: A pollo ad astra has an ingredient that is some cajun spice topping.
Original axiom: PolloAdAstra SubclassOf(hasIngredient some CajunSpiceTopping)
Sampled questions and axioms:
  - Question: Universe projects onto some immaterial three dimensional physical entity. True or false?
    Axiom: Universe SubclassOf(projectsOnto some ImmaterialThreeDimensionalPhysicalEntity)
  - Question: Does a has word conjunct have a child that is some token?
    Axiom: hasWordConjunct SubclassOf(hasChild some Token)
  - Question: Does a warthog eat some animal?
    Axiom: Warthog SubclassOf(eats some Animal)
  - Question: A dispersion colloid has stuff distribution that is some region. True or false?
    Axiom: DispersionColloid SubclassOf(hasStuffDistribution some Region)
  - Question: Does a cinema have a site that is some site?
    Axiom: Cinema SubclassOf(hasSite some Site)
  - Question: Is an apple part 

In [54]:
display(df.sample(2))

,axiom_pattern,axiom,template,question,question_type,op_category,ontology,file,question_type_short,op_category_short,row_id,seed,q_from_os,sample_axioms,sample_questions,dataset_seed
6958,[X] SubclassOf([prop] some [Y]),Cell SubclassOf(hasPatient some StructuredBiol...,What [prop/*OP_HAS_NOUNS-nounsNoArticle] does ...,What patient does a cell have?,What-1-part-1-rel,OP_HAS_NOUNS,BioTop,What-1-part-1-rel.csv,w1p1r,has_nouns,6958,3260950086,"[Pizza, Cultural-On, CopyrightAll, Stuff, AWO,...",[Mushroom SubclassOf(hasSpiciness only Spicine...,"[What spiciness does a mushroom have?, What do...",357
21529,"[Z] SubclassOf([X]),[Z] SubclassOf([prop] some...","Cell SubclassOf(Compound),Cell SubclassOf(isPa...",Which [X/noArticle] is [prop/*OP_IS_NOUNS_PREP...,Which compound is part of some universe?,What-2-part-1-rel-quant-some,OP_IS_NOUNS_PREP,BioTop,What-2-part-1-rel-quant-some.csv,w2p1r_qs,is_noun_prep,21529,4109306321,"[olia, CopyrightAll, AWO, Pizza, Cultural-On, ...","[hasWordConjunct SubclassOf(hasConjunct),hasWo...",[Which has conjunct has a child that is some t...,357


Now we save the dataset in the PARQUET format

In [55]:
import pyarrow.parquet as pq

filename = f"../dataset/qgenllm-ds-seed-{SEED}--{MAX_FEW_SHOTS}-few-shots.parquet"

df.to_parquet(
    filename,
    engine="pyarrow",
    index=False,
)

print("final dataset shape:", df.shape)
print("saved dataset:", filename)
print("all done - notebook ends here")


final dataset shape: (500, 16)
saved dataset: ../dataset/qgenllm-ds-seed-357--10-few-shots.parquet
all done - notebook ends here
